# Parte 2 do notebook: limpeza, divisão, transformação e pipeline

Continuação do `meta1.ipynb`. As células seguem a ordem da ficha 6 (divisão → transformações passo a passo → pipeline → ColumnTransformer).

**Antes de começar:** na primeira célula do notebook corrige o import para `import numpy as np` (está `numpy as numpy`). Aqui usa-se `np`.

In [1]:
# Célula de arranque. Se já tens estas variáveis no notebook, podes saltar esta célula.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("smoking.csv")

target = "smoking"
drop_cols = ["ID", "oral"]
cat_cols = ["gender", "tartar", "dental caries", "hearing(left)", "hearing(right)"]
ord_cols = ["Urine protein"]
num_cols = [c for c in df.columns
            if c not in cat_cols + ord_cols + drop_cols + [target]]
print(df.shape, len(num_cols))

(55692, 27) 18


## 4. Limpeza dos dados

### 4.1 Linhas duplicadas
O `ID` é único, por isso `df.duplicated()` não encontra nada. Temos de ignorar o `ID` e olhar só para as features.

In [2]:
features = [c for c in df.columns if c not in ["ID", target]]
print("Linhas repetidas (ignorando o ID):", df.duplicated(subset=features).sum())
print("Percentagem:", round(100 * df.duplicated(subset=features).mean(), 1), "%")

Linhas repetidas (ignorando o ID): 11140
Percentagem: 20.0 %


**Decisão (a confirmar no grupo):** remover os duplicados. Se ficarem, cópias da mesma linha caem em treino e em teste e a avaliação do M2 fica otimista.

In [3]:
df = df.drop_duplicates(subset=features).reset_index(drop=True)
print("Dimensão depois de remover duplicados:", df.shape)
print(df[target].value_counts(normalize=True).round(3))

Dimensão depois de remover duplicados: (44552, 27)
smoking
0    0.633
1    0.367
Name: proportion, dtype: float64


### 4.2 Colunas removidas (`ID` e `oral`)

In [4]:
print(df["oral"].value_counts())          # só tem o valor 'Y'
df = df.drop(columns=drop_cols)
print(df.shape)

oral
Y    44552
Name: count, dtype: int64
(44552, 25)


### 4.3 Valores impossíveis → NaN
Só os valores do grupo 2 (prováveis erros) são convertidos em `NaN`. Os outliers reais ficam como estão.

In [5]:
# Contagens antes da limpeza (para o relatório)
n_eye_l = (df["eyesight(left)"] == 9.9).sum()
n_eye_r = (df["eyesight(right)"] == 9.9).sum()
n_ldl   = (df["LDL"] > df["Cholesterol"]).sum()
n_hdl   = (df["HDL"] > 200).sum()
print("eyesight esq. = 9.9:", n_eye_l, "| dir. = 9.9:", n_eye_r)
print("LDL > colesterol total:", n_ldl, "| HDL > 200:", n_hdl)

eyesight esq. = 9.9: 74 | dir. = 9.9: 75
LDL > colesterol total: 19 | HDL > 200: 2


In [6]:
# 9,9 na visão é um código, não uma medição
df[["eyesight(left)", "eyesight(right)"]] = df[["eyesight(left)", "eyesight(right)"]].replace(9.9, np.nan)

# LDL maior que o colesterol total é impossível; HDL > 200 é implausível
df.loc[df["LDL"] > df["Cholesterol"], "LDL"] = np.nan
df.loc[df["HDL"] > 200, "HDL"] = np.nan

# Verificação: NaN por coluna
na = df.isna().sum()
na[na > 0]

eyesight(left)     74
eyesight(right)    75
HDL                 2
LDL                19
dtype: int64

## 7. Divisão em treino e teste

Fazemos a divisão **antes** de qualquer transformação que aprenda com os dados (ficha 6, passos 16 a 18). Usamos `stratify` para manter a proporção das classes. Substitui o `train_test_split` que tinhas no fim do notebook por este.

In [7]:
from sklearn.model_selection import train_test_split

x = df.drop(columns=[target])
y = df[target]

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=1, stratify=y)

print("Treino:", x_train.shape, "| Teste:", x_test.shape)
print("Proporção de fumadores no treino:", round(y_train.mean(), 3))
print("Proporção de fumadores no teste: ", round(y_test.mean(), 3))

Treino: (35641, 24) | Teste: (8911, 24)
Proporção de fumadores no treino: 0.367
Proporção de fumadores no teste:  0.367


## 6. Transformação e codificação (passo a passo, só no treino)

Como na ficha 6 (passos 19 a 24), primeiro mostramos cada transformação isolada e depois juntamo-las num pipeline.

### 6.1 Colunas numéricas: imputação pela mediana

In [8]:
from sklearn.impute import SimpleImputer

x_train_num = x_train[num_cols]
print("NaN antes:", x_train_num.isna().sum().sum())

imputer = SimpleImputer(strategy="median")
x_train_num_imp = pd.DataFrame(imputer.fit_transform(x_train_num),
                               columns=num_cols, index=x_train_num.index)
print("NaN depois:", x_train_num_imp.isna().sum().sum())

NaN antes: 139
NaN depois: 0


### 6.2 Colunas categóricas: One Hot Encoding

In [9]:
from sklearn.preprocessing import OneHotEncoder

x_train_cat = x_train[cat_cols]
ohe = OneHotEncoder(drop="if_binary", handle_unknown="ignore")
x_train_cat_enc = ohe.fit_transform(x_train_cat).toarray()

print(ohe.get_feature_names_out())
print(x_train_cat_enc.shape)

['gender_M' 'tartar_Y' 'dental caries_1' 'hearing(left)_2.0'
 'hearing(right)_2.0']
(35641, 5)


### 6.3 Coluna ordinal: Ordinal Encoding
A `Urine protein` tem uma ordem natural (1 < 2 < … < 6), por isso indicamos a ordem explicitamente.

In [10]:
from sklearn.preprocessing import OrdinalEncoder

x_train_ord = x_train[ord_cols]
oe = OrdinalEncoder(categories=[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]])
x_train_ord_enc = oe.fit_transform(x_train_ord)

print(np.unique(x_train_ord_enc, return_counts=True))

(array([0., 1., 2., 3., 4., 5.]), array([33648,  1150,   599,   179,    56,     9]))


### 6.4 Novo atributo: IMC (transformer próprio)
Como na ficha 6, o transformer recebe um array e indexa por posição de coluna.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class NewAttributes(BaseEstimator, TransformerMixin):
    def __init__(self, h_idx=0, w_idx=1):
        self.h_idx = h_idx
        self.w_idx = w_idx
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        bmi = X[:, self.w_idx] / (X[:, self.h_idx] / 100) ** 2
        return np.c_[X, bmi]
    def get_feature_names_out(self, input_features=None):
        return np.append(input_features, "BMI")

h = num_cols.index("height(cm)")
w = num_cols.index("weight(kg)")
adder = NewAttributes(h_idx=h, w_idx=w)
x_train_num_new = adder.fit_transform(x_train_num_imp.values)

print(x_train_num_imp.shape, "->", x_train_num_new.shape)
print("BMI (primeiros 5):", x_train_num_new[:5, -1].round(1))

## 7.2 Normalização

As variáveis têm escalas muito diferentes e seis delas (glicemia, triglicéridos, creatinina, AST, ALT e Gtp) têm forte assimetria à direita. Para essas aplicamos `log1p` antes do `StandardScaler`; as restantes só passam pelo `StandardScaler`. O scaler aprende a média e o desvio-padrão **só no treino**.

*(Alternativa mais simples: usar `RobustScaler` em vez de `log1p` + `StandardScaler`. Escolham uma e justifiquem no relatório.)*

In [ ]:
skew_cols = ["fasting blood sugar", "triglyceride", "serum creatinine", "AST", "ALT", "Gtp"]
num_normal_cols = [c for c in num_cols if c not in skew_cols]
print(len(num_normal_cols), "numéricas normais |", len(skew_cols), "assimétricas")

# Efeito do log1p na assimetria (skewness) — para o relatório
print(pd.DataFrame({"antes": x_train[skew_cols].skew(),
                    "depois do log1p": np.log1p(x_train[skew_cols]).skew()}).round(2))

## 8. Pipeline de pré-processamento (ColumnTransformer)

Um ramo por tipo de atributo, como no passo 4 da ficha 6.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer

h = num_normal_cols.index("height(cm)")
w = num_normal_cols.index("weight(kg)")

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("attr_adder", NewAttributes(h_idx=h, w_idx=w)),
    ("scaler", StandardScaler()),
])

skew_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
    ("scaler", StandardScaler()),
])

cat_pipeline = OneHotEncoder(drop="if_binary", handle_unknown="ignore")
ord_pipeline = OrdinalEncoder(categories=[[1.0, 2.0, 3.0, 4.0, 5.0, 6.0]])

preprocessing = ColumnTransformer([
    ("num",  num_pipeline,  num_normal_cols),
    ("skew", skew_pipeline, skew_cols),
    ("cat",  cat_pipeline,  cat_cols),
    ("ord",  ord_pipeline,  ord_cols),
])
preprocessing

Para ver o diagrama do pipeline no relatório, corre a célula acima numa sessão com `from sklearn import set_config; set_config(display="diagram")` (já é o comportamento por defeito no Jupyter/Colab).

### Aplicar ao treino (fit_transform) e ao teste (transform)

In [ ]:
x_train_prep = preprocessing.fit_transform(x_train)
x_test_prep  = preprocessing.transform(x_test)

print("Treino:", x_train_prep.shape, "| Teste:", x_test_prep.shape)
print("NaN no treino:", np.isnan(x_train_prep).sum(), "| NaN no teste:", np.isnan(x_test_prep).sum())

In [ ]:
cols_out = preprocessing.get_feature_names_out()
x_train_df = pd.DataFrame(x_train_prep, columns=cols_out, index=x_train.index)
x_test_df  = pd.DataFrame(x_test_prep,  columns=cols_out, index=x_test.index)
x_train_df.head()

In [ ]:
# Verificação: as colunas escaladas devem ter média ~0 e desvio-padrão ~1 no treino
x_train_df.describe().T[["mean", "std", "min", "max"]].round(2)

## Guardar os dados preparados (ficheiros .csv da entrega)

In [ ]:
df.to_csv("smoking_limpo.csv", index=False)

train_out = x_train_df.copy(); train_out[target] = y_train.values
test_out  = x_test_df.copy();  test_out[target]  = y_test.values
train_out.to_csv("smoking_treino_preparado.csv", index=False)
test_out.to_csv("smoking_teste_preparado.csv", index=False)
print(train_out.shape, test_out.shape)